# Chapter 8: RAG Generation — The Answer Layer
## Topic 8: RAG vs. Fine-Tuning — When Each Is the Right Answer

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- This chapter has spent seven topics building out RAG's generation layer in depth — prompt construction, citation, faithfulness, hallucination detection, streaming, multi-turn handling, query rewriting. This closing topic steps back and asks the question that should have been asked before any of that work began, and that any senior-level design review would ask: was RAG even the right architectural choice, versus fine-tuning a model directly on the domain?
- **RAG:** the model's weights stay frozen and general-purpose; domain knowledge is injected at inference time via retrieved context. The model "knows" about a domain only because relevant chunks are placed in its context window for each query.
- **Fine-tuning:** the model's weights are directly updated using labeled domain-specific training examples, so the model "knows" about the domain intrinsically, without needing retrieved context at inference time.
- Why this topic belongs at the end of a RAG-focused chapter rather than the beginning: understanding RAG's actual mechanics and limitations in depth is a prerequisite for evaluating this trade-off honestly — a superficial understanding of RAG makes "just fine-tune instead" sound like a simpler alternative, when in fact fine-tuning solves a different, narrower set of problems and introduces its own significant costs and limitations, several of which RAG is specifically chosen to avoid.


### 2. Internal Working — Step by Step, Framed as a Decision Framework

**The dimensions that actually determine the right choice:**

1. **Does the knowledge change frequently?** Policy documents, product terms, and rates can change — a new product launch, a regulatory change, a rate revision. RAG handles this by simply re-ingesting updated documents — the model's weights never need to change. Fine-tuning would require a full retraining cycle every time underlying facts change, which is operationally far more expensive and slower for genuinely dynamic content.
2. **Does the system need to cite and verify its sources?** Citation and faithfulness verification infrastructure exists specifically because RAG's context-injection architecture makes citation and verification possible — the model's context contains real, identifiable source documents it can be checked against. A fine-tuned model's "knowledge" is baked into its weights with no equivalent, checkable source to cite or verify against — a fine-tuned model cannot reliably cite which training example a specific claim came from.
3. **Is the domain vocabulary or reasoning style fundamentally different from the base model's training distribution?** This is the case fine-tuning is genuinely strong at — teaching a model to consistently produce output in a very specific format, tone, or reasoning pattern that prompting alone struggles to reliably enforce. Many domains don't obviously require this if the base model's general capabilities already handle reasoning and instruction-following well, and the actual domain-specific need is facts, which RAG handles more directly and verifiably than fine-tuning would.
4. **What is the actual measured gap, and where does it live?** This is the most important, and most often skipped, step. A real, measured accuracy gap between a prompted model and a domain-specific baseline might exist — but that gap should be measured before RAG was built to close it, and again after. The correct question before considering fine-tuning is: after retrieval, reranking, and verification infrastructure are all in place, where does the remaining gap actually live? If retrieval and generation with proper grounding is now closing most of that gap, fine-tuning may not be solving a problem that still exists.


### 3. How This Is Implemented in This Project — A Decision Framework, Not a Universal Answer

- The decision isn't a single yes/no choice but a structured framework: weigh whether knowledge changes frequently (favors RAG), whether citation and verification are required (favors RAG), and whether the actual measured gap is format/reasoning-related (favors fine-tuning) or fact-related (favors RAG).
- Applying this framework to a typical RAG use case: knowledge changes over time (favoring RAG), citation and verification are close to a compliance requirement in many regulated domains (favoring RAG), and the actual domain need is largely factual grounding rather than a fundamentally different reasoning style or output format (favoring RAG). Taken together, RAG is very often the primary architecture, with fine-tuning considered only for a narrower, specific purpose.
- The measured-gap step deserves special emphasis: any accuracy gap measured before a full RAG pipeline (retrieval, reranking, verification) existed reflects a knowledge-access problem, not necessarily a model-capability problem — and that gap should be re-measured after the full pipeline is built, not assumed to still exist at its original size.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **Security:** fine-tuning on sensitive data, if training examples include real user records, introduces a different exposure surface than RAG's approach — fine-tuned weights can potentially memorize and regurgitate specific training examples in ways that are harder to audit or redact after the fact than RAG's approach of filtering and access-scoping retrievable documents at query time. This is a meaningful, sometimes underappreciated argument in favor of RAG for a regulated domain, independent of the knowledge-freshness argument.
- **Deployment and monitoring:** RAG's failure modes are diagnosable and fixable at the content or retrieval layer without touching the model itself — an outdated policy document is a content-ingestion fix, not a model retrain. Fine-tuning's failure modes — a model that's learned something subtly wrong, or has degraded on tasks outside its fine-tuning distribution — require a new training run to fix, a fundamentally slower and more expensive remediation cycle. This is a genuine operational trade-off in favor of RAG's faster fix cycle for a system that needs to stay current and correctable.
- **Cost:** RAG incurs higher per-query inference cost (larger prompts due to retrieved context) but avoids the upfront and periodic cost of training runs. Fine-tuning incurs meaningful upfront and recurring training cost but can reduce per-query cost and latency once trained, since it needs less context injected at inference time.
- **Latency:** RAG's retrieval step adds latency on top of generation; fine-tuning's advantage, once trained, is that no retrieval step is needed at all, though this benefit doesn't apply if the underlying use case still fundamentally needs verifiable, up-to-date facts that only retrieval can guarantee are current.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **The core trade-off, stated plainly:** RAG trades higher per-query inference cost and added architectural complexity for verifiability, freshness, and faster remediation. Fine-tuning trades a slower, more expensive upfront or periodic training cost and reduced verifiability for potentially lower per-query cost and latency, and stronger control over output format, style, or reasoning patterns. Neither is universally superior — the right choice depends on which of these properties matters more for the specific use case.
- **Should a project ever fine-tune, and for what specifically?** Given this analysis, the most defensible fine-tuning use case for a RAG-based system, if pursued at all, is not replacing RAG's factual grounding, but rather improving the consistency of how the model uses retrieved context — for example, fine-tuning specifically on examples of correctly-cited, well-formatted, appropriately-hedged answers given retrieved context, so the base model's tendency to follow citation and faithfulness instructions becomes more reliable without needing ever-more-elaborate prompting. This is fine-tuning in service of the RAG architecture's reliability, not a replacement for it — a genuinely different design decision than "fine-tune instead of RAG."
- **The build-order dilemma:** doing this analysis at the end of a RAG-focused effort, after the infrastructure already exists, versus doing it before building any of that infrastructure, is itself a real decision with trade-offs — building RAG first and measuring its actual remaining gaps produces a more evidence-based fine-tuning decision than speculating about fine-tuning's value before RAG exists to compare against, but it does mean any fine-tuning decision comes later in the project timeline than it might if evaluated purely on paper upfront.


### 6. Alternatives and When to Use Each

- **RAG only:** the right choice when knowledge is dynamic, verifiability and citation are required (especially for a regulated domain), and the measured gap is primarily about knowledge access rather than format or reasoning style.
- **Fine-tuning only, no RAG:** appropriate for narrow, static-knowledge domains where verifiability is not a hard requirement, and where per-query latency and cost matter more than freshness or auditability.
- **RAG plus fine-tuning combined:** fine-tuning applied specifically to improve consistency of citation and faithfulness behavior on top of RAG's factual grounding, rather than replacing RAG's knowledge-access mechanism — often the most sophisticated and effective production pattern once basic RAG is mature and specific, measured gaps justify the added fine-tuning investment.
- **Prompt engineering alone, no RAG, no fine-tuning:** demonstrates a real ceiling in accuracy-sensitive domains — a real, measured gap against a strong baseline that motivates building RAG in the first place; prompting alone is often insufficient for domains with specific, high accuracy requirements.


### 7. Common Mistakes and Production Failures

- Treating "RAG vs fine-tuning" as a mutually exclusive, one-time architectural choice rather than a nuanced decision that can, and often should, combine both, applied to different specific problems.
- Considering fine-tuning before RAG's own gaps have been properly measured — fine-tuning to fix a problem that RAG, properly implemented, would have already solved is wasted effort.
- Fine-tuning on sensitive data without adequately considering the memorization and auditability risks that RAG's more contained, filterable, access-scoped context-injection approach avoids.
- Assuming fine-tuning solves a knowledge-freshness problem — it does not; fine-tuned knowledge is exactly as static as the training data it was created from, and updating it requires a full retraining cycle just as costly, often more so, as the fine-tuning was originally.
- Not accounting for the loss of citation and verifiability when moving any part of a system from RAG to fine-tuned-knowledge — this is not a minor detail for a regulated domain, it's close to a compliance requirement.


### 8. Lead-Level Interview Questions

**Basic**

- Q: What's the fundamental architectural difference between RAG and fine-tuning?
  A: RAG keeps the model's weights frozen and injects domain knowledge at inference time via retrieved context in the prompt. Fine-tuning directly updates the model's weights using labeled training examples, baking domain knowledge into the model itself, with no need for retrieved context at inference time.

- Q: Why does a regulated domain generally favor RAG over fine-tuning, even setting aside cost?
  A: Citation and faithfulness verification depend on being able to check a generated claim against a specific, identifiable source document — something RAG's context-injection architecture directly enables, since the retrieved chunks are real, traceable documents. A fine-tuned model's knowledge is embedded in its weights with no equivalent checkable source, making citation and auditability structurally much harder to guarantee.

**Intermediate**

- Q: A domain's underlying content changes over time — new products, rate revisions, regulatory updates. How does this specifically favor RAG's architecture?
  A: RAG handles knowledge updates by re-ingesting updated source documents into the knowledge base — no model retraining is required, and updates can take effect essentially as soon as ingestion completes. Fine-tuning bakes knowledge into model weights at training time; any subsequent change to the underlying facts requires a full retraining cycle to reflect the update, which is slower and more operationally expensive than a re-ingest approach, especially for content that changes with meaningful frequency.

- Q: A real, measured accuracy gap exists between a prompted model and a strong domain-specific baseline. Does this gap, by itself, justify fine-tuning?
  A: Not directly — if that gap was measured before a full RAG pipeline was built to address it, it largely reflects a knowledge-access problem, which is precisely what retrieval and context injection are designed to solve. The correct next step is re-measuring the gap after the full pipeline — retrieval, reranking, grounded generation, verification — is in place. Only if a meaningful gap remains after that, and specifically if that remaining gap is diagnosed as format, style, or reasoning-pattern related rather than fact-related, does fine-tuning become the well-justified next step rather than a premature one.

**Advanced**

- Q: Design a decision process to determine, with evidence rather than intuition, whether fine-tuning is worth pursuing after a full RAG pipeline is built and evaluated.
  A: First, ensure the full RAG pipeline is evaluated end-to-end using retrieval evaluation metrics and a faithfulness/relevance/correctness framework for generation quality, against a labeled evaluation set covering the real query distribution. Identify the specific, remaining gap — not a single aggregate number, but a breakdown by failure mode. If the remaining gap traces primarily to retrieval quality, the fix is in the retrieval pipeline, not fine-tuning. If it traces to faithfulness — the model not reliably following citation and grounding instructions even with good retrieved context — that's a strong signal fine-tuning specifically on well-formed, correctly-cited example outputs could improve consistency, which is fine-tuning in service of RAG, not a replacement of RAG's knowledge-access mechanism. If it traces to correctness — stale or wrong knowledge base content — neither RAG improvements nor fine-tuning fix it; the fix is knowledge base governance, entirely outside the RAG-vs-fine-tuning question.

**Scenario-based**

- Q: A stakeholder proposes fine-tuning as a faster path to better accuracy than continuing to refine the RAG pipeline. How do you respond?
  A: Before committing to fine-tuning, the remaining measured gap after the current RAG pipeline needs to be diagnosed by failure mode — is it a retrieval gap, a faithfulness gap, or a correctness gap? Fine-tuning is only well-justified for a faithfulness-related gap (inconsistent citation or grounding behavior); it does nothing for a correctness gap (stale knowledge base content) and duplicates effort already better solved by retrieval improvements for a retrieval gap. Without this diagnosis, fine-tuning risks becoming an expensive, slow-to-iterate investment that doesn't actually address the real remaining problem.

**Follow-up questions to expect:**

- "How would you measure whether fine-tuning for citation consistency actually improved things, versus just changing behavior in some other way?"
- "What would change about this analysis for a use case with genuinely static, rarely-changing knowledge?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **RAG and fine-tuning solve different classes of problems, not the same problem at different costs:** RAG solves a knowledge-access and verifiability problem; fine-tuning solves a behavior-consistency and output-format problem. Recognizing this distinction is what prevents treating them as simple substitutes for each other.
- **The measured-gap discipline generalizes beyond this specific decision:** the practice of re-measuring a problem after a partial fix is applied, rather than assuming the original measurement still holds, is a broadly applicable engineering discipline — many architectural decisions benefit from this same "measure again after the intervention" pattern.
- **Fine-tuning in service of an architecture, not as a replacement for it, is an underappreciated pattern:** using fine-tuning to improve how reliably a model follows a specific behavioral pattern (like citation discipline) on top of an existing architecture, rather than as a wholesale alternative architecture, is a more nuanced and often more practical use of fine-tuning than the simplistic "RAG vs fine-tuning" framing suggests.

### 10. Quick Revision Summary (for last-minute interview prep)

> RAG and fine-tuning solve different problems: RAG injects domain knowledge at inference time via retrieved context, keeping model weights frozen and knowledge verifiable and easily updatable; fine-tuning bakes domain knowledge or behavior patterns directly into model weights, trading verifiability and update-flexibility for potentially lower per-query cost and stronger control over output style. Dynamic knowledge, citation and verification requirements, and fact-centric domain needs all favor RAG. Fundamentally different output format, tone, or reasoning-pattern requirements are where fine-tuning is genuinely strong. The critical discipline is measuring the actual remaining gap after a full RAG pipeline is built and evaluated, rather than assuming a pre-RAG accuracy gap still justifies fine-tuning — and even then, the most defensible fine-tuning use case is usually improving consistency of behavior on top of RAG (like citation discipline), not replacing RAG's knowledge-access mechanism entirely. RAG and fine-tuning are not mutually exclusive, and the strongest production systems often combine both, each applied to the specific problem it's actually suited to solve.


### Module 1: The Decision Framework, Implemented

Turn the theory's four dimensions into an actual runnable decision function, not just a conceptual list.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: RAG vs fine-tuning decision framework, implemented
# ------------------------------------------------------------------

def evaluate_rag_vs_finetuning(
    knowledge_changes_frequently: bool,
    citation_and_verification_required: bool,
    measured_gap_after_full_rag_pipeline: float,   # 0.0 = no gap, 1.0 = huge gap
    gap_is_format_or_style_related: bool,           # vs. fact-related
    gap_is_reasoning_pattern_related: bool,          # vs. fact-related
    handles_sensitive_customer_data: bool,
) -> dict:
    """A structured decision framework -- weighs the real dimensions
    that determine whether RAG, fine-tuning, or both are appropriate.
    Returns reasons for and against fine-tuning, plus a recommendation."""

    reasons_against_finetuning = []
    reasons_for_finetuning = []

    if knowledge_changes_frequently:
        reasons_against_finetuning.append(
            "Frequent knowledge changes favor RAG's re-ingest-without-retrain model."
        )
    if citation_and_verification_required:
        reasons_against_finetuning.append(
            "Citation/faithfulness verification requires RAG's context-injection "
            "architecture; fine-tuned weights have no equivalent checkable source."
        )
    if handles_sensitive_customer_data:
        reasons_against_finetuning.append(
            "Fine-tuning on sensitive data risks memorization/regurgitation that is "
            "harder to audit than RAG's filterable, access-scoped retrieval."
        )
    if gap_is_format_or_style_related or gap_is_reasoning_pattern_related:
        reasons_for_finetuning.append(
            "A format/style/reasoning-pattern gap is exactly what fine-tuning is "
            "genuinely strong at addressing, unlike a pure knowledge gap."
        )
    if measured_gap_after_full_rag_pipeline < 0.1 and not (gap_is_format_or_style_related or gap_is_reasoning_pattern_related):
        reasons_against_finetuning.append(
            "Remaining gap after full RAG pipeline is small AND fact-related -- "
            "little evidence fine-tuning would address a problem that barely exists."
        )

    if len(reasons_for_finetuning) > 0 and measured_gap_after_full_rag_pipeline >= 0.1:
        recommendation = "CONSIDER FINE-TUNING (in service of RAG, not replacing it)"
    elif len(reasons_for_finetuning) > 0 and measured_gap_after_full_rag_pipeline < 0.1:
        recommendation = "MONITOR ONLY -- format/style gap exists but is currently small; revisit if it grows"
    else:
        recommendation = "RAG ONLY -- no evidence-based case for fine-tuning yet"

    return {
        "recommendation": recommendation,
        "reasons_against_finetuning": reasons_against_finetuning,
        "reasons_for_finetuning": reasons_for_finetuning,
    }


print("=" * 70)
print("MODULE 1: DECISION FRAMEWORK DEFINED")
print("=" * 70)
print("Framework ready. Testing against three realistic scenarios in Module 2.")


MODULE 1: DECISION FRAMEWORK DEFINED
Framework ready. Testing against three realistic scenarios in Module 2.


### Module 2: Testing the Framework Against Three Realistic Scenarios

Run the decision function against scenarios representing a typical regulated-domain case, a static-knowledge case, and a genuine citation-consistency case.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: Testing the framework against realistic scenarios
# ------------------------------------------------------------------

SCENARIOS = [
    {
        "label": "Scenario A: Regulated financial RAG system (dynamic policy, needs citations)",
        "params": dict(
            knowledge_changes_frequently=True,
            citation_and_verification_required=True,
            measured_gap_after_full_rag_pipeline=0.05,   # small remaining gap after full RAG
            gap_is_format_or_style_related=False,
            gap_is_reasoning_pattern_related=False,
            handles_sensitive_customer_data=True,
        ),
    },
    {
        "label": "Scenario B: Static internal documentation bot, no verification requirement",
        "params": dict(
            knowledge_changes_frequently=False,
            citation_and_verification_required=False,
            measured_gap_after_full_rag_pipeline=0.15,
            gap_is_format_or_style_related=False,
            gap_is_reasoning_pattern_related=False,
            handles_sensitive_customer_data=False,
        ),
    },
    {
        "label": "Scenario C: Citation-consistency gap after full RAG pipeline built",
        "params": dict(
            knowledge_changes_frequently=True,
            citation_and_verification_required=True,
            measured_gap_after_full_rag_pipeline=0.22,   # meaningful remaining gap
            gap_is_format_or_style_related=True,          # model inconsistently follows citation format
            gap_is_reasoning_pattern_related=False,
            handles_sensitive_customer_data=True,
        ),
    },
]

print("=" * 70)
print("DECISION FRAMEWORK APPLIED TO THREE SCENARIOS")
print("=" * 70)

for scenario in SCENARIOS:
    result = evaluate_rag_vs_finetuning(**scenario["params"])
    print(f"\n[{scenario['label']}]")
    print(f"  RECOMMENDATION: {result['recommendation']}")
    if result["reasons_against_finetuning"]:
        print("  Reasons against fine-tuning:")
        for r in result["reasons_against_finetuning"]:
            print(f"    - {r}")
    if result["reasons_for_finetuning"]:
        print("  Reasons for fine-tuning:")
        for r in result["reasons_for_finetuning"]:
            print(f"    - {r}")

print("\nNotice Scenario A (dynamic, needs citations, small remaining gap) lands")
print("on RAG-only -- exactly what the theory argues for a regulated domain.")
print("Scenario C (citation-consistency gap specifically, meaningful size)")
print("is the one case that reaches CONSIDER FINE-TUNING -- and even then,")
print("the framework explicitly recommends it IN SERVICE OF RAG, not instead of it.")
print("\nModule 2 complete. Run Module 3 next.")


DECISION FRAMEWORK APPLIED TO THREE SCENARIOS

[Scenario A: Regulated financial RAG system (dynamic policy, needs citations)]
  RECOMMENDATION: RAG ONLY -- no evidence-based case for fine-tuning yet
  Reasons against fine-tuning:
    - Frequent knowledge changes favor RAG's re-ingest-without-retrain model.
    - Citation/faithfulness verification requires RAG's context-injection architecture; fine-tuned weights have no equivalent checkable source.
    - Fine-tuning on sensitive data risks memorization/regurgitation that is harder to audit than RAG's filterable, access-scoped retrieval.
    - Remaining gap after full RAG pipeline is small AND fact-related -- little evidence fine-tuning would address a problem that barely exists.

[Scenario B: Static internal documentation bot, no verification requirement]
  RECOMMENDATION: RAG ONLY -- no evidence-based case for fine-tuning yet

[Scenario C: Citation-consistency gap after full RAG pipeline built]
  RECOMMENDATION: CONSIDER FINE-TUNING (i

### Module 3: The Measured-Gap Discipline — Before vs After RAG

Demonstrates the theory's most important point concretely: a pre-RAG accuracy gap can shrink dramatically once RAG is actually built, changing the fine-tuning decision entirely.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: The measured-gap discipline -- before vs after RAG
#
# This demonstrates the theory's central warning: a gap measured BEFORE
# a full RAG pipeline exists does NOT automatically justify fine-tuning,
# because RAG itself may close most of that gap. Only the REMAINING
# gap, measured AFTER, should inform the fine-tuning decision.
# ------------------------------------------------------------------

# illustrative before/after numbers showing how a knowledge-access gap
# largely closes once retrieval + grounding is actually built
PIPELINE_STAGES = [
    ("Prompted model alone, no retrieval", 0.65),
    ("+ BM25 keyword retrieval", 0.78),
    ("+ Dense retrieval (hybrid)", 0.86),
    ("+ Reranking", 0.91),
    ("+ Citation + hallucination verification", 0.93),
]

BASELINE_TARGET = 0.97  # the domain-specific baseline being compared against

print("=" * 70)
print("MEASURED GAP AT EACH STAGE OF BUILDING THE RAG PIPELINE")
print("=" * 70)
print(f"Target baseline: {BASELINE_TARGET}\n")

for stage_name, measured_score in PIPELINE_STAGES:
    gap = BASELINE_TARGET - measured_score
    print(f"  {stage_name:<45} | score={measured_score:.2f} | gap={gap:.2f}")

initial_gap = BASELINE_TARGET - PIPELINE_STAGES[0][1]
final_gap = BASELINE_TARGET - PIPELINE_STAGES[-1][1]
gap_closed_percent = (1 - final_gap / initial_gap) * 100

print(f"\nInitial gap (before any RAG): {initial_gap:.2f}")
print(f"Remaining gap (after full RAG pipeline): {final_gap:.2f}")
print(f"RAG closed {gap_closed_percent:.0f}% of the original gap.")

print("\nIf fine-tuning had been decided based on the INITIAL 0.32 gap,")
print("that decision would have been made against a problem that RAG itself")
print("was about to mostly solve. The remaining 0.04 gap after the full")
print("pipeline is a MUCH weaker case for fine-tuning -- and per Module 2's")
print("framework, whether it justifies fine-tuning further depends on WHERE")
print("that remaining gap lives (format/consistency vs fact-related), not")
print("just its size.")

print("\nModule 3 complete. All Chapter 8 Topic 8 modules done.")
print()
print("=" * 70)
print("CHAPTER 8 TOPIC 8 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  RAG and fine-tuning solve DIFFERENT problems: RAG solves knowledge
  access + verifiability; fine-tuning solves behavior/format consistency.

  Dynamic knowledge + citation requirements + fact-centric gaps all
  favor RAG. Format/style/reasoning-pattern gaps favor fine-tuning.

  ALWAYS re-measure the gap AFTER building the full RAG pipeline --
  a pre-RAG gap can shrink dramatically once retrieval + grounding
  actually exist, changing the fine-tuning decision entirely.

  The most defensible fine-tuning use case (if any) is usually
  IN SERVICE OF RAG -- improving citation/faithfulness consistency --
  not REPLACING RAG's knowledge-access mechanism.

  RAG vs fine-tuning is not mutually exclusive -- the strongest systems
  often combine both, each applied to the specific problem it solves.
""")


MEASURED GAP AT EACH STAGE OF BUILDING THE RAG PIPELINE
Target baseline: 0.97

  Prompted model alone, no retrieval            | score=0.65 | gap=0.32
  + BM25 keyword retrieval                      | score=0.78 | gap=0.19
  + Dense retrieval (hybrid)                    | score=0.86 | gap=0.11
  + Reranking                                   | score=0.91 | gap=0.06
  + Citation + hallucination verification       | score=0.93 | gap=0.04

Initial gap (before any RAG): 0.32
Remaining gap (after full RAG pipeline): 0.04
RAG closed 88% of the original gap.

If fine-tuning had been decided based on the INITIAL 0.32 gap,
that decision would have been made against a problem that RAG itself
was about to mostly solve. The remaining 0.04 gap after the full
pipeline is a MUCH weaker case for fine-tuning -- and per Module 2's
framework, whether it justifies fine-tuning further depends on WHERE
that remaining gap lives (format/consistency vs fact-related), not
just its size.

Module 3 complete. All C